In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

data = [
  (1, "Alice",  "Engineering", 95000, "2021-03-15", "F"),
  (2, "Bob",    "Marketing",   72000, "2019-07-01", "M"),
  (3, "Carol",  "Engineering", 88000, "2020-11-20", "F"),
  (4, "David",  "HR",          61000, "2022-01-10", "M"),
  (5, "Eve",    "Marketing",   79000, "2018-05-25", "F"),
  (6, "Frank",  "Engineering", 102000,"2017-09-30", "M"),
  (7, "Grace",  "HR",          58000, "2023-02-14", "F"),
]

schema = StructType([
  StructField("id",         IntegerType(), False),
  StructField("name",       StringType(),  True),
  StructField("dept",       StringType(),  True),
  StructField("salary",     IntegerType(), True),
  StructField("join_date",  StringType(),  True),
  StructField("gender",     StringType(),  True),
])

df = spark.createDataFrame(data, schema)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

data2 = [
    ("Engineering", "Bangalore"),
    ("Marketing", "Mumbai"),
    ("HR", "Chennai"),
    ("Finance", "Delhi")   # extra to show join behavior
]

schema2 = StructType([
    StructField("dept", StringType(), True),
    StructField("location", StringType(), True)
])

df2 = spark.createDataFrame(data2, schema2)

In [0]:
df.display()

id,name,dept,salary,join_date,gender
1,Alice,Engineering,95000,2021-03-15,F
2,Bob,Marketing,72000,2019-07-01,M
3,Carol,Engineering,88000,2020-11-20,F
4,David,HR,61000,2022-01-10,M
5,Eve,Marketing,79000,2018-05-25,F
6,Frank,Engineering,102000,2017-09-30,M
7,Grace,HR,58000,2023-02-14,F


In [0]:
df2.display()

dept,location
Engineering,Bangalore
Marketing,Mumbai
HR,Chennai
Finance,Delhi


In [0]:
"""
1.Create a second DataFrame with columns dept and location. Inner join it with employees on dept.
2. Do a left join with the location DataFrame. Show which employees have no location match.
3. Create a manager DataFrame with dept and manager. Join employees with it and show only employees whose salary is higher than 80000, along with their manager.
4.Create a dept_location DataFrame with dept and city. Inner join it with employees on dept and show name, dept, salary, city.
5. Left join employees with the location DataFrame. Show all employees even if they have no matching location.
6.After the left join, filter and show only employees who have NO location match (i.e.location is null).
7.Right join employees with the location DataFrame. Show all locations even if no employee belongs to that dept.
8.Using a semi join, show only the employees whose dept exists in the location DataFrame (no location columns in result).
9.Using an anti join, show only the employees whose dept does NOT exist in the location DataFrame.
10.Create a dept_gender_bonus DataFrame with dept, gender, and extra_bonus. Join it with employees on both dept and gender. Show name, dept, gender, salary, extra_bonus.
 
Union:
1.Create a second DataFrame with 2 new employees using the same schema and union it with the original.
2.Union two DataFrames by column name (not position) and remove any duplicate rows.
3.Union the original DataFrame with itself, then count how many times each employee name appears.
"""

In [0]:
# 1.Create a second DataFrame with columns dept and location. Inner join it with employees on dept.
df.join(df2, on = "dept", how ="inner").display()

dept,id,name,salary,join_date,gender,location
Engineering,1,Alice,95000,2021-03-15,F,Bangalore
Marketing,2,Bob,72000,2019-07-01,M,Mumbai
Engineering,3,Carol,88000,2020-11-20,F,Bangalore
HR,4,David,61000,2022-01-10,M,Chennai
Marketing,5,Eve,79000,2018-05-25,F,Mumbai
Engineering,6,Frank,102000,2017-09-30,M,Bangalore
HR,7,Grace,58000,2023-02-14,F,Chennai


In [0]:
# 2. Do a left join with the location DataFrame. Show which employees have no location match.
df.join(df2, on ="dept", how="left").display()

dept,id,name,salary,join_date,gender,location
Engineering,1,Alice,95000,2021-03-15,F,Bangalore
Marketing,2,Bob,72000,2019-07-01,M,Mumbai
Engineering,3,Carol,88000,2020-11-20,F,Bangalore
HR,4,David,61000,2022-01-10,M,Chennai
Marketing,5,Eve,79000,2018-05-25,F,Mumbai
Engineering,6,Frank,102000,2017-09-30,M,Bangalore
HR,7,Grace,58000,2023-02-14,F,Chennai


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

data3 = [
    ("Engineering", "Sowmiya"),
    ("Marketing", "Shivarnjini"),
    ("HR", "Rithika")
]

managerSchema = StructType([ 
      StructField("dept", StringType(), True),
      StructField("manager", StringType(), True)
    ])
df_manager = spark.createDataFrame(data3, managerSchema)

In [0]:
df_manager.display()

dept,manager
Engineering,Sowmiya
Marketing,Shivarnjini
HR,Rithika


In [0]:
# 3. Create a manager DataFrame with dept and manager. Join employees with it and show only employees whose salary is higher than 80000, along with their manager.
from pyspark.sql.functions import col
df.join(df_manager, on = "dept", how="inner").filter(col("salary")>80000).display()

dept,id,name,salary,join_date,gender,manager
Engineering,1,Alice,95000,2021-03-15,F,Sowmiya
Engineering,3,Carol,88000,2020-11-20,F,Sowmiya
Engineering,6,Frank,102000,2017-09-30,M,Sowmiya


In [0]:
dept_location = df2
dept_location.display()

dept,location
Engineering,Bangalore
Marketing,Mumbai
HR,Chennai
Finance,Delhi


In [0]:
# 4.Create a dept_location DataFrame with dept and city. Inner join it with employees on dept and show name, dept, salary, city.
df.join(dept_location, on = "dept", how ="inner").select("name", "dept", "salary", "location").display()


name,dept,salary,location
Alice,Engineering,95000,Bangalore
Bob,Marketing,72000,Mumbai
Carol,Engineering,88000,Bangalore
David,HR,61000,Chennai
Eve,Marketing,79000,Mumbai
Frank,Engineering,102000,Bangalore
Grace,HR,58000,Chennai


In [0]:
# 5. Left join employees with the location DataFrame. Show all employees even if they have no matching location.
df.join(dept_location, on="dept", how="left").display()

dept,id,name,salary,join_date,gender,location
Engineering,1,Alice,95000,2021-03-15,F,Bangalore
Marketing,2,Bob,72000,2019-07-01,M,Mumbai
Engineering,3,Carol,88000,2020-11-20,F,Bangalore
HR,4,David,61000,2022-01-10,M,Chennai
Marketing,5,Eve,79000,2018-05-25,F,Mumbai
Engineering,6,Frank,102000,2017-09-30,M,Bangalore
HR,7,Grace,58000,2023-02-14,F,Chennai


In [0]:
from pyspark.sql.types import StructType, StructField, StringType

data3 = [
    ("Engineering", "Bangalore"),
    ("Marketing", "Chennai"),
]

schema3 = StructType([
    StructField("dept", StringType(), True),
    StructField("location", StringType(), True)
])

df3 = spark.createDataFrame(data3,schema3)

In [0]:
df3.display()

dept,location
Engineering,Bangalore
Marketing,Chennai


In [0]:
# 6.After the left join, filter and show only employees who have NO location match (i.e.location is null).
df.join(df3, on="dept", how="left").filter(col("location").isNull()).display()

dept,id,name,salary,join_date,gender,location
HR,4,David,61000,2022-01-10,M,null
HR,7,Grace,58000,2023-02-14,F,null


In [0]:
# 7.Right join employees with the location DataFrame. Show all locations even if no employee belongs to that dept.
df.join(dept_location, on="dept", how="right").display()

dept,id,name,salary,join_date,gender,location
Engineering,6,Frank,102000,2017-09-30,M,Bangalore
Engineering,3,Carol,88000,2020-11-20,F,Bangalore
Engineering,1,Alice,95000,2021-03-15,F,Bangalore
Marketing,5,Eve,79000,2018-05-25,F,Mumbai
Marketing,2,Bob,72000,2019-07-01,M,Mumbai
HR,7,Grace,58000,2023-02-14,F,Chennai
HR,4,David,61000,2022-01-10,M,Chennai
Finance,null,null,null,null,null,Delhi


In [0]:
# 8.Using a semi join, show only the employees whose dept exists in the location DataFrame (no location columns in result).
df.join(dept_location, on="dept", how="left_semi").display()

dept,id,name,salary,join_date,gender
Engineering,1,Alice,95000,2021-03-15,F
Marketing,2,Bob,72000,2019-07-01,M
Engineering,3,Carol,88000,2020-11-20,F
HR,4,David,61000,2022-01-10,M
Marketing,5,Eve,79000,2018-05-25,F
Engineering,6,Frank,102000,2017-09-30,M
HR,7,Grace,58000,2023-02-14,F


In [0]:
# 9.Using an anti join, show only the employees whose dept does NOT exist in the location DataFrame.
df.join(dept_location, on="dept", how="left_anti").display()

dept,id,name,salary,join_date,gender


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

data_bonus = [
    ("Engineering", "Female", 5000),
    ("Engineering", "Male", 4000),
    ("Marketing", "Female", 3000),
    ("Marketing", "M", 3500),
    ("HR", "Female", 2000),
    ("HR", "Male", 2500)
]

schema_bonus = StructType([
    StructField("dept", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("extra_bonus", IntegerType(), True)
])

df_bonus = spark.createDataFrame(data_bonus, schema_bonus)

In [0]:
display(df_bonus)

dept,gender,extra_bonus
Engineering,Female,5000
Engineering,Male,4000
Marketing,Female,3000
Marketing,M,3500
HR,Female,2000
HR,Male,2500


In [0]:
df.join(df_bonus, on = ["dept", "gender"], how="full").display()

dept,gender,id,name,salary,join_date,extra_bonus
Engineering,F,1,Alice,95000,2021-03-15,null
Marketing,M,2,Bob,72000,2019-07-01,3500
Engineering,F,3,Carol,88000,2020-11-20,null
HR,M,4,David,61000,2022-01-10,null
Marketing,F,5,Eve,79000,2018-05-25,null
Engineering,M,6,Frank,102000,2017-09-30,null
HR,F,7,Grace,58000,2023-02-14,null
HR,Female,null,null,null,null,2000
HR,Male,null,null,null,null,2500
Engineering,Male,null,null,null,null,4000


In [0]:
new_data = [
    (8, "Henry", "Finance", 67000, "2022-06-10", "M"),
    (9, "Ivy", "Engineering", 91000, "2021-12-01", "F"),
    (6, "Frank",  "Engineering", 102000,"2017-09-30", "M"),
    (7, "Grace",  "HR",          58000, "2023-02-14", "F"),
]

df_new = spark.createDataFrame(new_data, schema)

In [0]:
df_new.display()

id,name,dept,salary,join_date,gender
8,Henry,Finance,67000,2022-06-10,M
9,Ivy,Engineering,91000,2021-12-01,F
6,Frank,Engineering,102000,2017-09-30,M
7,Grace,HR,58000,2023-02-14,F


In [0]:

#Union: 1.Create a second DataFrame with 2 new employees using the same schema and union it with the original.
df.union(df_new).display()

id,name,dept,salary,join_date,gender
1,Alice,Engineering,95000,2021-03-15,F
2,Bob,Marketing,72000,2019-07-01,M
3,Carol,Engineering,88000,2020-11-20,F
4,David,HR,61000,2022-01-10,M
5,Eve,Marketing,79000,2018-05-25,F
6,Frank,Engineering,102000,2017-09-30,M
7,Grace,HR,58000,2023-02-14,F
8,Henry,Finance,67000,2022-06-10,M
9,Ivy,Engineering,91000,2021-12-01,F
6,Frank,Engineering,102000,2017-09-30,M


In [0]:
#2.Union two DataFrames by column name (not position) and remove any duplicate rows.
df.unionByName(df_new).dropDuplicates().display()

id,name,dept,salary,join_date,gender
1,Alice,Engineering,95000,2021-03-15,F
2,Bob,Marketing,72000,2019-07-01,M
3,Carol,Engineering,88000,2020-11-20,F
4,David,HR,61000,2022-01-10,M
5,Eve,Marketing,79000,2018-05-25,F
6,Frank,Engineering,102000,2017-09-30,M
7,Grace,HR,58000,2023-02-14,F
8,Henry,Finance,67000,2022-06-10,M
9,Ivy,Engineering,91000,2021-12-01,F


In [0]:
# 3.Union the original DataFrame with itself, then count how many times each employee name appears.
df.union(df).groupBy("name").count().display()

name,count
Alice,2
Bob,2
Carol,2
David,2
Eve,2
Frank,2
Grace,2
